# FORGE M01 — Kaggle Environment Debug Probe

**Purpose:** Environment probe and diagnostic only.  
**Version:** M01  
**Repository:** https://github.com/m-cahill/forge

---

**⚠️ IMPORTANT NOTICE ⚠️**

This is an **environment probe only**. It does NOT:
- Produce a scored submission
- Prove Kaggle submission readiness
- Train any model
- Validate adapter packages
- Claim any leaderboard score

See `docs/kaggle/notebook_debug_standard.md` for FORGE notebook standards.

## 1. Run Mode Declaration

In [ ]:
import os

# Detect run mode from Kaggle environment
RUN_TYPE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "Interactive")
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

print(f"Run Mode: {RUN_TYPE}")
print(f"Is Kaggle: {IS_KAGGLE}")
print()
print("Mode meanings:")
print("  Interactive - Debugging only; does not prove submission")
print("  Commit - Notebook commit; does not alone prove leaderboard score")
print("  Submit/scored - Competition scoring path; requires full evidence")
print()
print("This notebook is for ENVIRONMENT PROBE only.")

## 2. Python / Platform / CWD / sys.path

In [ ]:
import platform
import sys

print("=" * 60)
print("PYTHON / PLATFORM INFO")
print("=" * 60)
print(f"Python version: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Machine: {platform.machine()}")
print(f"Processor: {platform.processor()}")
print(f"CWD: {os.getcwd()}")
print()
print("sys.path (first 10):")
for i, p in enumerate(sys.path[:10]):
    print(f"  [{i}] {p}")
if len(sys.path) > 10:
    print(f"  ... and {len(sys.path) - 10} more")

## 3. Kaggle Environment Variables

In [ ]:
print("=" * 60)
print("KAGGLE ENVIRONMENT VARIABLES")
print("=" * 60)

KAGGLE_VARS = [
    "KAGGLE_KERNEL_RUN_TYPE",
    "KAGGLE_URL_BASE",
    "KAGGLE_DATA_PROXY_TOKEN",
    "KAGGLE_USER_SECRETS_TOKEN",
    "KAGGLE_CONTAINER_NAME",
    "KAGGLE_KERNEL_INTEGRATIONS",
]

for var in KAGGLE_VARS:
    value = os.environ.get(var)
    if value:
        # Mask tokens/secrets
        if "TOKEN" in var or "SECRET" in var:
            display_value = f"{value[:8]}..." if len(value) > 8 else "[SET]"
        else:
            display_value = value
        print(f"{var}: {display_value}")
    else:
        print(f"{var}: [NOT SET]")

## 4. Kaggle Filesystem Paths

In [ ]:
print("=" * 60)
print("KAGGLE FILESYSTEM")
print("=" * 60)

KAGGLE_PATHS = [
    "/kaggle",
    "/kaggle/input",
    "/kaggle/working",
    "/kaggle/temp",
]

for path in KAGGLE_PATHS:
    exists = os.path.exists(path)
    is_dir = os.path.isdir(path) if exists else False
    print(f"\n{path}:")
    print(f"  Exists: {exists}")
    print(f"  Is directory: {is_dir}")

    if exists and is_dir:
        try:
            contents = os.listdir(path)
            print(f"  Contents ({len(contents)} items):")
            for item in contents[:15]:
                item_path = os.path.join(path, item)
                is_subdir = os.path.isdir(item_path)
                print(f"    {'[DIR]' if is_subdir else '[FILE]'} {item}")
            if len(contents) > 15:
                print(f"    ... and {len(contents) - 15} more")
        except PermissionError:
            print("  [Permission denied]")

## 5. Disk Free Space

In [ ]:
import shutil

print("=" * 60)
print("DISK SPACE")
print("=" * 60)

paths_to_check = ["/kaggle/working", "/kaggle", os.getcwd(), "/"]

for path in paths_to_check:
    if os.path.exists(path):
        try:
            usage = shutil.disk_usage(path)
            total_gb = usage.total / (1024**3)
            used_gb = usage.used / (1024**3)
            free_gb = usage.free / (1024**3)
            pct_used = (usage.used / usage.total) * 100
            print(f"\n{path}:")
            print(f"  Total: {total_gb:.2f} GB")
            print(f"  Used:  {used_gb:.2f} GB ({pct_used:.1f}%)")
            print(f"  Free:  {free_gb:.2f} GB")
            break  # Only show first available
        except Exception as e:
            print(f"{path}: Error - {e}")

## 6. GPU Visibility / nvidia-smi

In [ ]:
import subprocess

print("=" * 60)
print("GPU VISIBILITY")
print("=" * 60)

# Check CUDA_VISIBLE_DEVICES
cuda_visible = os.environ.get("CUDA_VISIBLE_DEVICES", "[NOT SET]")
print(f"CUDA_VISIBLE_DEVICES: {cuda_visible}")

# Try nvidia-smi
print("\nnvidia-smi output:")
try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free,driver_version", "--format=csv"],
        capture_output=True,
        text=True,
        timeout=10,
    )
    if result.returncode == 0:
        print(result.stdout)
    else:
        print(f"nvidia-smi failed: {result.stderr}")
except FileNotFoundError:
    print("nvidia-smi not found (no GPU or drivers not installed)")
except subprocess.TimeoutExpired:
    print("nvidia-smi timed out")
except Exception as e:
    print(f"Error running nvidia-smi: {e}")

## 7. Torch / CUDA Import and Version

In [ ]:
print("=" * 60)
print("TORCH / CUDA")
print("=" * 60)

try:
    import torch

    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")

    if torch.cuda.is_available():
        print(f"CUDA version: {torch.version.cuda}")
        print(f"cuDNN version: {torch.backends.cudnn.version()}")
        print(f"Device count: {torch.cuda.device_count()}")
        for i in range(torch.cuda.device_count()):
            print(f"  Device {i}: {torch.cuda.get_device_name(i)}")
            props = torch.cuda.get_device_properties(i)
            print(f"    Memory: {props.total_memory / (1024**3):.2f} GB")
except ImportError:
    print("PyTorch not installed")
except Exception as e:
    print(f"Error checking PyTorch/CUDA: {e}")

## 8. Competition / Model / Dataset Path Discovery

In [ ]:
print("=" * 60)
print("COMPETITION / MODEL / DATASET PATHS")
print("=" * 60)

# Common competition-related paths
COMPETITION_PATHS = [
    "/kaggle/input/nvidia-nemotron-model-reasoning-challenge",
    "/kaggle/input/nemotron-3-nano-30b-a3b-bf16",
    "/kaggle/input/nvidia-nemotron-3-nano-30b-a3b-bf16",
]

for path in COMPETITION_PATHS:
    exists = os.path.exists(path)
    print(f"\n{path}:")
    print(f"  Exists: {exists}")

    if exists and os.path.isdir(path):
        try:
            contents = os.listdir(path)
            print(f"  Contents ({len(contents)} items):")
            for item in contents[:10]:
                item_path = os.path.join(path, item)
                if os.path.isfile(item_path):
                    size = os.path.getsize(item_path)
                    size_str = (
                        f"{size / (1024**2):.2f} MB"
                        if size > 1024 * 1024
                        else f"{size / 1024:.2f} KB"
                    )
                    print(f"    [FILE] {item} ({size_str})")
                else:
                    print(f"    [DIR]  {item}")
            if len(contents) > 10:
                print(f"    ... and {len(contents) - 10} more")
        except PermissionError:
            print("  [Permission denied]")

# Also check /kaggle/input for any other datasets
if os.path.exists("/kaggle/input"):
    print("\nAll datasets in /kaggle/input:")
    for item in os.listdir("/kaggle/input"):
        print(f"  {item}")

## 9. Output Artifact Test Path

In [ ]:
print("=" * 60)
print("OUTPUT ARTIFACT TEST")
print("=" * 60)

# Create a test output directory (for debug only, NOT submission)
if IS_KAGGLE:
    TEST_OUTPUT_DIR = "/kaggle/working/tmp/forge_debug"
else:
    TEST_OUTPUT_DIR = "./tmp/forge_debug"

print(f"Test output directory: {TEST_OUTPUT_DIR}")

try:
    os.makedirs(TEST_OUTPUT_DIR, exist_ok=True)
    print(f"  Created: {os.path.exists(TEST_OUTPUT_DIR)}")

    # Write a test file
    test_file = os.path.join(TEST_OUTPUT_DIR, "probe_test.txt")
    with open(test_file, "w") as f:
        f.write("FORGE M01 Debug Probe\n")
        f.write(f"Timestamp: {__import__('datetime').datetime.utcnow().isoformat()}Z\n")
        f.write(f"Run type: {RUN_TYPE}\n")

    print(f"  Test file written: {test_file}")
    print(f"  File size: {os.path.getsize(test_file)} bytes")

except Exception as e:
    print(f"  Error: {e}")

print("\n⚠️ This is a debug output, NOT a submission artifact.")

## 10. SHA256 Helper

In [ ]:
import hashlib


def compute_sha256(filepath: str) -> str:
    """Compute SHA256 hash of a file."""
    sha256_hash = hashlib.sha256()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            sha256_hash.update(chunk)
    return sha256_hash.hexdigest()


print("=" * 60)
print("SHA256 HELPER")
print("=" * 60)
print("SHA256 helper function defined: compute_sha256(filepath)")
print("Use this to hash submission.zip and other artifacts.")

# Example with test file if it exists
test_file = os.path.join(TEST_OUTPUT_DIR, "probe_test.txt")
if os.path.exists(test_file):
    hash_value = compute_sha256(test_file)
    print(f"\nExample - {test_file}:")
    print(f"  SHA256: {hash_value}")

## 11. Timing Helper

In [ ]:
import time
from contextlib import contextmanager


@contextmanager
def timer(description: str):
    """Context manager for timing code blocks."""
    start = time.perf_counter()
    yield
    elapsed = time.perf_counter() - start
    print(f"[TIMER] {description}: {elapsed:.3f}s")


print("=" * 60)
print("TIMING HELPER")
print("=" * 60)
print("Timer context manager defined.")
print("Usage:")
print('  with timer("my operation"):')
print("      # code to time")

# Example
print("\nExample:")
with timer("sleep 0.1s"):
    time.sleep(0.1)

## 12. Exception / Traceback Helper

In [ ]:
import traceback


def safe_run(func, *args, **kwargs):
    """Run a function and capture any exceptions with full traceback."""
    try:
        return func(*args, **kwargs)
    except Exception as e:
        print(f"[ERROR] {type(e).__name__}: {e}")
        print("\nFull traceback:")
        traceback.print_exc()
        return None


print("=" * 60)
print("EXCEPTION / TRACEBACK HELPER")
print("=" * 60)
print("safe_run(func, *args, **kwargs) - runs function with traceback capture")

# Example
print("\nExample (intentional error):")


def demo_error():
    return 1 / 0


result = safe_run(demo_error)
print(f"\nResult: {result}")

## 13. Summary

In [ ]:
from datetime import datetime

print("=" * 60)
print("PROBE SUMMARY")
print("=" * 60)
print(f"Timestamp (UTC): {datetime.utcnow().isoformat()}Z")
print(f"Run type: {RUN_TYPE}")
print(f"Is Kaggle: {IS_KAGGLE}")
print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")

# GPU summary
try:
    import torch

    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("GPU: None available")
except ImportError:
    print("GPU: PyTorch not installed")

print()
print("=" * 60)
print("⚠️ THIS IS AN ENVIRONMENT PROBE ONLY")
print("⚠️ IT DOES NOT PRODUCE A SCORED SUBMISSION")
print("⚠️ IT DOES NOT PROVE KAGGLE SUBMISSION READINESS")
print("=" * 60)